
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.


# Similarity Robustness Experiment
## Is the proposed gain merely an artifact of choosing a weak similarity baseline?

This notebook is the **second and final reviewer-oriented experiment**.

The scientific question is:

> Does Future-Compatible retrieval remain advantageous when Pattern retrieval is replaced by several conventional notions of time-series similarity?

We evaluate the same frozen confirmatory queries under:

1. **Raw Cosine**
2. **Pattern / Pearson** — the current baseline
3. **Negative L2**
4. **Spectral Cosine**
5. **SARAF-Matched**
6. **Future-Compatible Learned (Ours)**

### Important note on Pattern vs Pearson

The current Pattern score is cosine similarity after mean removal:

\[
s_{\mathrm{pat}}(q,i)
=
\frac{
    (x_q-\bar x_q)^\top(x_i-\bar x_i)
}{
    \|x_q-\bar x_q\|_2
    \|x_i-\bar x_i\|_2
}.
\]

For one-dimensional windows, this is mathematically the same ranking as **Pearson correlation**. Therefore, we do **not** report Pearson as a separate duplicated column; the paper should label this baseline as:

> **Pattern / Pearson**

This is also well aligned with RAFT, which compares Pearson correlation, cosine similarity, cosine with projection, and negative L2, and selects Pearson as its default retrieval similarity.

### Spectral baseline

To test a qualitatively different notion of similarity, we additionally compare the **magnitude spectrum**:

\[
\phi_{\mathrm{spec}}(x)
=
\log\!\left(
1+\left|
\operatorname{rFFT}(x-\bar x)
\right|
\right),
\]

excluding the DC term, and use cosine similarity between spectral features.

This baseline is phase-insensitive and emphasizes periodic/frequency content rather than point-wise temporal alignment.

---

## Frozen protocol

- Electricity, Traffic, Exchange, Solar
- \(L=96\)
- \(H\in\{24,48,96\}\)
- Same-channel retrieval
- The same temporally admissible **test memory** as the confirmatory experiment
- The same fixed train-scale windows and future targets
- \(K=10\)
- Existing Ours and SARAF-Matched outputs are reused
- 5,000-replicate moving-block bootstrap

### Why rebuild the conventional similarities from the full admissible memory?

The proposed reranker starts from a Pattern Top-\(M\) pool. If alternative similarities were evaluated only *inside that same Pattern Top-\(M\) pool*, they would be artificially constrained by the very baseline we are testing.

This notebook therefore reconstructs the frozen test memory and lets each conventional similarity retrieve its own Top-\(K\) neighbors from that same memory.

A validation cell checks that reconstructing Pattern/Pearson from this memory reproduces the cached confirmatory Pattern Top-\(M\) candidates. If this check fails, the notebook stops instead of silently running an unmatched experiment.


---

### Cache implementation note

The final confirmatory cache stores `pattern`, `context`, and `future`,
but not the original past trajectory. This fixed notebook therefore
reconstructs raw past windows from the original dataset files using the
same train-only normalization, channel selection, and cached
`ChannelIndex`/`Anchor` metadata. It then verifies that centered cosine
reproduces the cached Pattern Top-\(M\) retrieval before evaluating the
alternative similarities.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Imports and frozen paths

In [ ]:

from pathlib import Path
import math
import warnings
import random

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_ROOT = REPO_DATA_ROOT

DATA_PATHS = {
    "Electricity":
        DATA_ROOT / "electricity/electricity.csv",

    "Traffic":
        DATA_ROOT / "traffic/traffic.csv",

    "Exchange":
        DATA_ROOT / "exchange_rate/exchange_rate.csv",

    "Solar":
        DATA_ROOT / "Solar/solar_AL.txt",
}

RESULT_DIR = REPO_WORK_ROOT / "final_confirmatory"

CACHE_DIR = RESULT_DIR / "cache"

SARAF_DIR = (
    RESULT_DIR /
    "external_baselines" /
    "saraf_matched"
)

OUT_DIR = (
    RESULT_DIR /
    "similarity_robustness"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DATASETS = [
    "Electricity",
    "Traffic",
    "Exchange",
    "Solar",
]

HORIZONS = [24, 48, 96]

SEQ_LEN = 96
TOP_M = 100
TOP_K = 10

MAX_MEMORY_WINDOWS = 50000
BLOCK_ANCHORS = 10
N_BOOT = 5000
EPS = 1e-8

# Exact confirmatory settings.
MAX_CHANNELS = {
    "Electricity": 32,
    "Traffic": 32,
    "Exchange": None,
    "Solar": None,
}

DATASET_SEED = {
    "Electricity": 3303,
    "Traffic": 4404,
    "Exchange": 5505,
    "Solar": 6606,
}

QUERY_BATCH = 256

print("Device:", DEVICE)
print("Confirmatory directory:", RESULT_DIR)
print("Output directory:", OUT_DIR)

assert RESULT_DIR.exists()
assert CACHE_DIR.exists()
assert SARAF_DIR.exists(), (
    "Run the SARAF-Matched notebook first."
)

for name, path in DATA_PATHS.items():
    assert path.exists(), (name, path)


## 1. Verify required files and inspect cache keys

In [ ]:

required = [
    RESULT_DIR / "00_data_manifest.csv",
    RESULT_DIR / "09_main_confirmatory_summary.csv",
]

for dataset_name in DATASETS:
    for H in HORIZONS:
        required.extend([
            CACHE_DIR / f"{dataset_name}_H{H}_windows.npz",
            CACHE_DIR / f"{dataset_name}_H{H}_meta.csv.gz",
            CACHE_DIR / f"{dataset_name}_H{H}_same_topM.npz",
            RESULT_DIR / f"query_level_{dataset_name}_H{H}.csv.gz",
            SARAF_DIR / f"query_level_{dataset_name}_H{H}.csv.gz",
        ])

missing = [str(p) for p in required if not p.exists()]

if missing:
    print("Missing files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError(
        "Required frozen outputs are missing."
    )

for dataset_name in DATASETS:
    H = HORIZONS[0]

    w = np.load(
        CACHE_DIR / f"{dataset_name}_H{H}_windows.npz"
    )

    t = np.load(
        CACHE_DIR / f"{dataset_name}_H{H}_same_topM.npz"
    )

    print(
        dataset_name,
        "| windows keys:",
        list(w.keys()),
        "| topM keys:",
        list(t.keys()),
    )


## 2. Temporal split and deterministic balanced memory sampling

In [ ]:

manifest = pd.read_csv(
    RESULT_DIR / "00_data_manifest.csv"
)

N_ROWS = {
    row["Dataset"]: int(row["Rows"])
    for _, row in manifest.iterrows()
    if row["Dataset"] in DATASETS
}

assert set(N_ROWS) == set(DATASETS)


def split_boundaries(n):
    train_end = int(0.70 * n)
    val_end = int(0.80 * n)

    return {
        "train_end": train_end,
        "val_end": val_end,
    }


SPLITS = {
    d: split_boundaries(N_ROWS[d])
    for d in DATASETS
}


def balanced_subset(
    meta,
    indices,
    max_n,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(indices) <= max_n:
        return np.sort(indices)

    rng = np.random.default_rng(seed)

    channels = (
        meta.iloc[indices]["ChannelIndex"]
        .to_numpy(dtype=np.int64)
    )

    unique_channels = np.unique(channels)
    random_channel_order = rng.permutation(
        unique_channels
    )

    base = max_n // len(unique_channels)
    extra = max_n % len(unique_channels)

    chosen_parts = []

    for rank, c in enumerate(random_channel_order):
        pos = indices[channels == c]
        quota = base + (
            1 if rank < extra else 0
        )
        take = min(quota, len(pos))

        if take > 0:
            chosen_parts.append(
                rng.choice(
                    pos,
                    size=take,
                    replace=False,
                )
            )

    chosen = np.unique(
        np.concatenate(chosen_parts)
    )

    if len(chosen) < max_n:
        remaining = np.setdiff1d(
            indices,
            chosen,
            assume_unique=False,
        )

        add_n = min(
            max_n - len(chosen),
            len(remaining),
        )

        if add_n > 0:
            chosen = np.concatenate([
                chosen,
                rng.choice(
                    remaining,
                    size=add_n,
                    replace=False,
                ),
            ])

    return np.sort(
        chosen.astype(np.int64)
    )



## 3. Recover the exact frozen test memory

If the Top-\(M\) cache directly stores test-memory indices, they are used.

Otherwise, the notebook reconstructs the temporally valid test memory:

\[
\mathrm{FutureEnd}(i) < t_{\mathrm{val\_end}}
\]

and searches the small deterministic seed-offset range used by the confirmatory pipeline.

The selected memory must contain **all cached Pattern Top-\(M\) candidates**. A later retrieval-level validation then verifies that centered cosine reproduces the cached candidate set.


In [ ]:

def infer_test_memory(
    dataset_name,
    H,
    meta,
    topm,
):
    """
    Reconstruct the exact frozen test-memory sample used by the
    final confirmatory notebook.

    Confirmatory rule:
        base = DATASET_SEED[dataset] + H * 10
        test_memory seed = base + 5
    """
    val_end = SPLITS[
        dataset_name
    ]["val_end"]

    future_end = (
        meta["FutureEnd"]
        .to_numpy(dtype=np.int64)
    )

    full_test_memory = np.where(
        future_end < val_end
    )[0].astype(np.int64)

    base = (
        DATASET_SEED[dataset_name] +
        H * 10
    )

    memory = balanced_subset(
        meta,
        full_test_memory,
        MAX_MEMORY_WINDOWS,
        base + 5,
    )

    cached_candidates = np.unique(
        topm["test_idx"]
        .astype(np.int64)
        .reshape(-1)
    )

    coverage = float(
        np.isin(
            cached_candidates,
            memory,
        ).mean()
    )

    if coverage < 0.999999:
        raise RuntimeError(
            f"Exact test-memory reconstruction failed for "
            f"{dataset_name}, H={H}. "
            f"Cached-candidate coverage={coverage:.8f}."
        )

    return memory, {
        "Mode": "exact_confirmatory_seed",
        "SeedOffset": 5,
        "CachedCandidateCoverage": coverage,
    }



## 4. Reconstruct raw past windows from the original datasets

The confirmatory cache intentionally stores only:

```text
pattern, context, future
```

and not the raw past window. This is sufficient for the original
Pattern reranking experiment, but Raw Cosine, Euclidean/L2, and spectral
retrieval require the actual past trajectory.

We therefore reconstruct the exact past windows from the original
datasets using:

- the same train-only channel normalization,
- the same deterministic channel selection,
- the cached `ChannelIndex` and `Anchor`.

The cached `pattern` array is still used indirectly as a validation
target: reconstructed centered-cosine retrieval must reproduce the
cached Pattern Top-100 pool.


In [ ]:

def load_standard_csv(path):
    df = pd.read_csv(path)

    timestamp_cols = [
        c for c in df.columns
        if str(c).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }
    ]

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    good_cols = [
        c for c in x.columns
        if x[c].notna().mean() > 0.99
    ]

    x = x[good_cols]

    x = (
        x
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert x.shape[1] > 0
    assert np.isfinite(
        x.to_numpy(dtype=np.float32)
    ).all()

    return x


def load_solar_txt(path):
    x = pd.read_csv(
        path,
        header=None,
    )

    if x.shape[1] == 1:
        x = pd.read_csv(
            path,
            header=None,
            sep=r"\s+",
        )

    x = (
        x
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert x.shape[1] == 137

    x.columns = [
        f"Solar_{i:03d}"
        for i in range(x.shape[1])
    ]

    return x


RAW = {
    "Electricity":
        load_standard_csv(
            DATA_PATHS["Electricity"]
        ),

    "Traffic":
        load_standard_csv(
            DATA_PATHS["Traffic"]
        ),

    "Exchange":
        load_standard_csv(
            DATA_PATHS["Exchange"]
        ),

    "Solar":
        load_solar_txt(
            DATA_PATHS["Solar"]
        ),
}


SELECTED_CHANNELS = {}
CHANNEL_NORMALIZED = {}

for dataset_name in DATASETS:
    df = RAW[dataset_name]

    train_end = SPLITS[
        dataset_name
    ]["train_end"]

    train = df.iloc[:train_end]

    train_std = train.std(
        axis=0,
        ddof=0,
    )

    valid_cols = [
        c for c in df.columns
        if (
            np.isfinite(train_std[c])
            and train_std[c] > 1e-6
        )
    ]

    max_c = MAX_CHANNELS[
        dataset_name
    ]

    if (
        max_c is not None
        and len(valid_cols) > max_c
    ):
        idx = np.linspace(
            0,
            len(valid_cols) - 1,
            max_c,
            dtype=int,
        )

        selected = [
            valid_cols[i]
            for i in idx
        ]
    else:
        selected = valid_cols

    SELECTED_CHANNELS[
        dataset_name
    ] = selected

    df_sel = df[selected]

    mu = (
        df_sel.iloc[:train_end]
        .mean(axis=0)
        .to_numpy(dtype=np.float32)
    )

    sd = (
        df_sel.iloc[:train_end]
        .std(axis=0, ddof=0)
        .to_numpy(dtype=np.float32)
    )

    assert np.all(sd > 1e-6)

    arr = df_sel.to_numpy(
        dtype=np.float32
    )

    z = (
        arr -
        mu[None, :]
    ) / sd[None, :]

    assert np.isfinite(z).all()

    CHANNEL_NORMALIZED[
        dataset_name
    ] = z.astype(np.float32)

    print(
        dataset_name,
        "| reconstructed channels:",
        len(selected),
        "| rows:",
        len(z),
    )


def extract_past_windows(
    dataset_name,
    meta,
    indices,
):
    """
    Reconstruct [N, SEQ_LEN] train-normalized past windows exactly
    from cached metadata.
    """
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    sub = meta.iloc[
        indices
    ]

    channels = sub[
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    anchors = sub[
        "Anchor"
    ].to_numpy(
        dtype=np.int64
    )

    out = np.empty(
        (
            len(indices),
            SEQ_LEN,
        ),
        dtype=np.float32,
    )

    for j, (
        c,
        anchor,
    ) in enumerate(
        zip(
            channels,
            anchors,
        )
    ):
        out[j] = arr[
            anchor - SEQ_LEN:
            anchor,
            c,
        ]

    return out


TASK = {}
memory_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        key = (
            dataset_name,
            H,
        )

        w = np.load(
            CACHE_DIR /
            f"{dataset_name}_H{H}_windows.npz"
        )

        meta = pd.read_csv(
            CACHE_DIR /
            f"{dataset_name}_H{H}_meta.csv.gz"
        )

        topm = np.load(
            CACHE_DIR /
            f"{dataset_name}_H{H}_same_topM.npz"
        )

        # Cache is intentionally compact.
        assert set(
            [
                "pattern",
                "context",
                "future",
            ]
        ).issubset(
            set(w.files)
        )

        q_idx = np.asarray(
            topm["test_query"],
            dtype=np.int64,
        )

        cached_topm_idx = np.asarray(
            topm["test_idx"],
            dtype=np.int64,
        )

        cached_topm_score = np.asarray(
            topm["test_score"],
            dtype=np.float32,
        )

        memory_idx, memory_info = infer_test_memory(
            dataset_name,
            H,
            meta,
            topm,
        )

        # Reconstruct only the frozen query and memory windows, not all
        # cached windows. This keeps memory use modest.
        q_past = extract_past_windows(
            dataset_name,
            meta,
            q_idx,
        )

        mem_past = extract_past_windows(
            dataset_name,
            meta,
            memory_idx,
        )

        future_all = np.asarray(
            w["future"],
            dtype=np.float32,
        )

        q_future = future_all[
            q_idx
        ].copy()

        mem_future = future_all[
            memory_idx
        ].copy()

        q_channel = (
            meta.iloc[q_idx]["ChannelIndex"]
            .to_numpy(dtype=np.int64)
        )

        q_anchor = (
            meta.iloc[q_idx]["Anchor"]
            .to_numpy(dtype=np.int64)
        )

        mem_channel = (
            meta.iloc[memory_idx]["ChannelIndex"]
            .to_numpy(dtype=np.int64)
        )

        # Map global cached-window index -> row in mem_past/mem_future.
        global_to_memory = np.full(
            len(meta),
            -1,
            dtype=np.int64,
        )

        global_to_memory[
            memory_idx
        ] = np.arange(
            len(memory_idx),
            dtype=np.int64,
        )

        TASK[key] = {
            "q_idx": q_idx,
            "q_past": q_past,
            "q_future": q_future,
            "q_channel": q_channel,
            "q_anchor": q_anchor,

            "memory_idx": memory_idx,
            "memory_past": mem_past,
            "memory_future": mem_future,
            "memory_channel": mem_channel,
            "global_to_memory": global_to_memory,

            "cached_topm_idx": cached_topm_idx,
            "cached_topm_score": cached_topm_score,
        }

        memory_rows.append({
            "Dataset": dataset_name,
            "Horizon": H,
            "Queries": len(q_idx),
            "MemoryWindows": len(memory_idx),
            **memory_info,
        })

        # Explicitly release the full future cache before the next task.
        del future_all
        w.close()
        topm.close()

memory_table = pd.DataFrame(
    memory_rows
)

display(memory_table)

memory_table.to_csv(
    OUT_DIR /
    "00_memory_reconstruction.csv",
    index=False,
)


## 5. Similarity feature definitions

In [ ]:

def l2_normalize_np(x):
    denom = np.linalg.norm(
        x,
        axis=1,
        keepdims=True,
    )

    return (
        x /
        np.maximum(
            denom,
            EPS,
        )
    ).astype(np.float32)


def raw_cosine_feature(x):
    return l2_normalize_np(
        x.astype(np.float32)
    )


def pearson_feature(x):
    centered = (
        x -
        x.mean(
            axis=1,
            keepdims=True,
        )
    )

    return l2_normalize_np(
        centered.astype(np.float32)
    )


def offset_l2_feature(x):
    # Level offset is removed, but relative shape/amplitude remain.
    return (
        x -
        x[:, -1:]
    ).astype(np.float32)


def spectral_feature(x):
    centered = (
        x -
        x.mean(
            axis=1,
            keepdims=True,
        )
    )

    mag = np.abs(
        np.fft.rfft(
            centered,
            axis=1,
        )
    )

    # Remove DC. It should already be near zero after centering.
    mag = mag[:, 1:]

    feat = np.log1p(
        mag
    ).astype(np.float32)

    return l2_normalize_np(feat)



## 6. GPU retrieval from the full frozen memory

For cosine-type metrics we use a normalized dot product.

For Negative L2:

\[
s_{\mathrm{L2}}(q,i)
=
-\frac{1}{L}
\left\|
(x_q-x_q^{(L)})
-
(x_i-x_i^{(L)})
\right\|_2^2 .
\]

Each query can retrieve only from memory windows belonging to the same channel.


In [ ]:

def topk_cosine_gpu(
    q_feat,
    m_feat,
    k,
    batch_size=QUERY_BATCH,
):
    q_feat = np.asarray(
        q_feat,
        dtype=np.float32,
    )

    m_feat = np.asarray(
        m_feat,
        dtype=np.float32,
    )

    m = torch.from_numpy(
        m_feat
    ).to(DEVICE)

    all_idx = []
    all_score = []

    for start in range(
        0,
        len(q_feat),
        batch_size,
    ):
        q = torch.from_numpy(
            q_feat[
                start:
                start + batch_size
            ]
        ).to(DEVICE)

        score = q @ m.T

        val, idx = torch.topk(
            score,
            k=k,
            dim=1,
            largest=True,
            sorted=True,
        )

        all_idx.append(
            idx.cpu().numpy()
        )

        all_score.append(
            val.cpu().numpy()
        )

    return (
        np.concatenate(
            all_idx,
            axis=0,
        ),
        np.concatenate(
            all_score,
            axis=0,
        ),
    )


def topk_negative_l2_gpu(
    q_feat,
    m_feat,
    k,
    batch_size=QUERY_BATCH,
):
    q_feat = np.asarray(
        q_feat,
        dtype=np.float32,
    )

    m_feat = np.asarray(
        m_feat,
        dtype=np.float32,
    )

    m = torch.from_numpy(
        m_feat
    ).to(DEVICE)

    m_sq = (
        m.pow(2)
        .sum(dim=1)
        [None, :]
    )

    L = float(
        m_feat.shape[1]
    )

    all_idx = []
    all_score = []

    for start in range(
        0,
        len(q_feat),
        batch_size,
    ):
        q = torch.from_numpy(
            q_feat[
                start:
                start + batch_size
            ]
        ).to(DEVICE)

        q_sq = (
            q.pow(2)
            .sum(dim=1)
            [:, None]
        )

        dist2 = (
            q_sq +
            m_sq -
            2.0 * (
                q @ m.T
            )
        )

        score = -dist2 / L

        val, idx = torch.topk(
            score,
            k=k,
            dim=1,
            largest=True,
            sorted=True,
        )

        all_idx.append(
            idx.cpu().numpy()
        )

        all_score.append(
            val.cpu().numpy()
        )

    return (
        np.concatenate(
            all_idx,
            axis=0,
        ),
        np.concatenate(
            all_score,
            axis=0,
        ),
    )


## 7. Validate that Pattern/Pearson reconstructs the cached Pattern Top-M

In [ ]:

@torch.no_grad()
def retrieve_metric(
    task,
    metric,
    k,
):
    q_past = task[
        "q_past"
    ]

    memory_past = task[
        "memory_past"
    ]

    q_channel = task[
        "q_channel"
    ]

    memory_idx = task[
        "memory_idx"
    ]

    memory_channel = task[
        "memory_channel"
    ]

    selected_global = np.empty(
        (
            len(q_past),
            k,
        ),
        dtype=np.int64,
    )

    selected_score = np.empty(
        (
            len(q_past),
            k,
        ),
        dtype=np.float32,
    )

    channels = np.unique(
        q_channel
    )

    for c in channels:
        q_pos = np.where(
            q_channel == c
        )[0]

        mem_pos = np.where(
            memory_channel == c
        )[0]

        if len(mem_pos) < k:
            raise RuntimeError(
                f"Channel {c} has only "
                f"{len(mem_pos)} memory windows."
            )

        q_raw = q_past[
            q_pos
        ]

        m_raw = memory_past[
            mem_pos
        ]

        if metric == "RawCosine":
            q_feat = raw_cosine_feature(
                q_raw
            )

            m_feat = raw_cosine_feature(
                m_raw
            )

            local_idx, local_score = (
                topk_cosine_gpu(
                    q_feat,
                    m_feat,
                    k,
                )
            )

        elif metric == "PatternPearson":
            q_feat = pearson_feature(
                q_raw
            )

            m_feat = pearson_feature(
                m_raw
            )

            local_idx, local_score = (
                topk_cosine_gpu(
                    q_feat,
                    m_feat,
                    k,
                )
            )

        elif metric == "NegativeL2":
            q_feat = offset_l2_feature(
                q_raw
            )

            m_feat = offset_l2_feature(
                m_raw
            )

            local_idx, local_score = (
                topk_negative_l2_gpu(
                    q_feat,
                    m_feat,
                    k,
                )
            )

        elif metric == "SpectralCosine":
            q_feat = spectral_feature(
                q_raw
            )

            m_feat = spectral_feature(
                m_raw
            )

            local_idx, local_score = (
                topk_cosine_gpu(
                    q_feat,
                    m_feat,
                    k,
                )
            )

        else:
            raise ValueError(metric)

        selected_global[
            q_pos
        ] = memory_idx[
            mem_pos[
                local_idx
            ]
        ]

        selected_score[
            q_pos
        ] = local_score

    return (
        selected_global,
        selected_score,
    )


validation_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        task = TASK[
            (
                dataset_name,
                H
            )
        ]

        reconstructed_idx, reconstructed_score = (
            retrieve_metric(
                task,
                "PatternPearson",
                TOP_M,
            )
        )

        cached_idx = task[
            "cached_topm_idx"
        ]

        recalls = []

        for a, b in zip(
            reconstructed_idx,
            cached_idx,
        ):
            recalls.append(
                len(
                    np.intersect1d(
                        a,
                        b,
                    )
                )
                /
                TOP_M
            )

        mean_recall = float(
            np.mean(recalls)
        )

        exact_top10 = float(
            np.mean(
                np.all(
                    reconstructed_idx[
                        :,
                        :TOP_K
                    ]
                    ==
                    cached_idx[
                        :,
                        :TOP_K
                    ],
                    axis=1,
                )
            )
        )

        validation_rows.append({
            "Dataset": dataset_name,
            "Horizon": H,
            "MeanRecallAt100_vsCached": mean_recall,
            "ExactTop10Fraction": exact_top10,
        })

        print(
            dataset_name,
            H,
            "| Recall@100 vs cache:",
            round(mean_recall, 6),
            "| exact Top-10 fraction:",
            round(exact_top10, 6),
        )

        if mean_recall < 0.995:
            raise RuntimeError(
                f"Pattern reconstruction mismatch for "
                f"{dataset_name}, H={H}: "
                f"Recall@100={mean_recall:.6f}. "
                "Stop here rather than comparing unmatched memories."
            )

validation_table = pd.DataFrame(
    validation_rows
)

display(validation_table)

validation_table.to_csv(
    OUT_DIR /
    "01_pattern_reconstruction_check.csv",
    index=False,
)


## 8. Evaluate the four conventional similarities

In [ ]:

def selected_query_metrics(
    task,
    selected_idx,
):
    local_pos = task[
        "global_to_memory"
    ][
        selected_idx
    ]

    if np.any(
        local_pos < 0
    ):
        raise RuntimeError(
            "Selected candidate is not in the frozen test memory."
        )

    selected_future = task[
        "memory_future"
    ][
        local_pos
    ]

    q_future = task[
        "q_future"
    ]

    analog = np.mean(
        (
            selected_future -
            q_future[
                :,
                None,
                :
            ]
        ) ** 2,
        axis=2,
    ).mean(
        axis=1
    )

    forecast = np.mean(
        (
            selected_future.mean(
                axis=1
            )
            -
            q_future
        ) ** 2,
        axis=1,
    )

    return (
        analog.astype(
            np.float32
        ),
        forecast.astype(
            np.float32
        ),
    )


SIMILARITY_METHODS = [
    "RawCosine",
    "PatternPearson",
    "NegativeL2",
    "SpectralCosine",
]

QUERY_RESULTS = {}
summary_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        key = (
            dataset_name,
            H,
        )

        task = TASK[
            (
                dataset_name,
                H
            )
        ]

        existing = pd.read_csv(
            RESULT_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz"
        )

        saraf = pd.read_csv(
            SARAF_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz"
        )

        assert len(existing) == len(
            task[
                "q_idx"
            ]
        )

        assert len(saraf) == len(
            existing
        )

        q = pd.DataFrame({
            "Anchor":
                task[
                    "q_anchor"
                ],

            "ChannelIndex":
                task[
                    "q_channel"
                ],

            "PatternPearson_AnalogFutureMSE":
                existing[
                    "Pattern_AnalogFutureMSE"
                ].to_numpy(),

            "PatternPearson_RetrievalForecastMSE":
                existing[
                    "Pattern_RetrievalForecastMSE"
                ].to_numpy(),

            "SARAFMatched_AnalogFutureMSE":
                saraf[
                    "SARAF_AnalogFutureMSE"
                ].to_numpy(),

            "SARAFMatched_RetrievalForecastMSE":
                saraf[
                    "SARAF_UniformForecastMSE"
                ].to_numpy(),

            "Learned_AnalogFutureMSE":
                existing[
                    "Learned_AnalogFutureMSE"
                ].to_numpy(),

            "Learned_RetrievalForecastMSE":
                existing[
                    "Learned_RetrievalForecastMSE"
                ].to_numpy(),
        })

        for method in [
            "RawCosine",
            "NegativeL2",
            "SpectralCosine",
        ]:
            selected_idx, _ = retrieve_metric(
                task,
                method,
                TOP_K,
            )

            analog, forecast = (
                selected_query_metrics(
                    task,
                    selected_idx,
                )
            )

            q[
                f"{method}_AnalogFutureMSE"
            ] = analog

            q[
                f"{method}_RetrievalForecastMSE"
            ] = forecast

        QUERY_RESULTS[
            key
        ] = q

        for method in [
            "RawCosine",
            "PatternPearson",
            "NegativeL2",
            "SpectralCosine",
            "SARAFMatched",
            "Learned",
        ]:
            summary_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Method":
                    method,

                "AnalogFutureMSE":
                    float(
                        q[
                            f"{method}_AnalogFutureMSE"
                        ].mean()
                    ),

                "RetrievalForecastMSE":
                    float(
                        q[
                            f"{method}_RetrievalForecastMSE"
                        ].mean()
                    ),
            })

        q.to_csv(
            OUT_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz",
            index=False,
            compression="gzip",
        )

summary = pd.DataFrame(
    summary_rows
)

display(
    summary.sort_values(
        [
            "Dataset",
            "Horizon",
            "AnalogFutureMSE",
        ]
    )
)

summary.to_csv(
    OUT_DIR /
    "02_similarity_summary.csv",
    index=False,
)


## 9. Paper-ready wide table

In [ ]:

method_order = [
    "RawCosine",
    "PatternPearson",
    "NegativeL2",
    "SpectralCosine",
    "SARAFMatched",
    "Learned",
]

paper_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        x = (
            summary[
                (summary["Dataset"] == dataset_name) &
                (summary["Horizon"] == H)
            ]
            .set_index("Method")
        )

        row = {
            "Dataset":
                dataset_name,

            "Horizon":
                H,
        }

        for method in method_order:
            row[method] = float(
                x.loc[
                    method,
                    "AnalogFutureMSE",
                ]
            )

        conventional = [
            row["RawCosine"],
            row["PatternPearson"],
            row["NegativeL2"],
            row["SpectralCosine"],
        ]

        row[
            "BestConventionalSimilarity"
        ] = min(
            conventional
        )

        row[
            "Ours_vs_BestConventional_%"
        ] = (
            100.0 *
            (
                row[
                    "BestConventionalSimilarity"
                ]
                -
                row[
                    "Learned"
                ]
            )
            /
            row[
                "BestConventionalSimilarity"
            ]
        )

        paper_rows.append(
            row
        )

paper_table = pd.DataFrame(
    paper_rows
)

display(paper_table)

paper_table.to_csv(
    OUT_DIR / "03_paper_similarity_table.csv",
    index=False,
)



### Important interpretation of `BestConventionalSimilarity`

`BestConventionalSimilarity` is the **post-hoc lower envelope** of the four conventional similarities on each test task.

It is intentionally a conservative diagnostic:

> Even if one were allowed to look back after evaluation and choose whichever conventional similarity happened to work best on that task, how does Ours compare?

It must **not** be described as a deployable validation-selected baseline unless a separate validation selection experiment is performed.


## 10. Moving-block bootstrap: Ours vs every similarity baseline

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(x)

    assert n >= block_len

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    means = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        parts = []

        for _ in range(
            n_blocks
        ):
            start = int(
                rng.integers(
                    0,
                    max_start + 1,
                )
            )

            parts.append(
                x[
                    start:
                    start +
                    block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[:n]

        means[b] = sample.mean()

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    means,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    means,
                    0.975,
                )
            ),

        "SignificantImprovement":
            bool(
                np.quantile(
                    means,
                    0.025,
                ) > 0
            ),
    }


BASELINES_FOR_BOOTSTRAP = [
    "RawCosine",
    "PatternPearson",
    "NegativeL2",
    "SpectralCosine",
    "SARAFMatched",
]

bootstrap_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        q = QUERY_RESULTS[
            (
                dataset_name,
                H,
            )
        ]

        for baseline in BASELINES_FOR_BOOTSTRAP:
            diff = (
                q[
                    f"{baseline}_AnalogFutureMSE"
                ]
                -
                q[
                    "Learned_AnalogFutureMSE"
                ]
            )

            tmp = pd.DataFrame({
                "Anchor":
                    q[
                        "Anchor"
                    ],

                "Diff":
                    diff,
            })

            anchor_diff = (
                tmp
                .groupby(
                    "Anchor"
                )[
                    "Diff"
                ]
                .mean()
                .sort_index()
                .to_numpy()
            )

            result = moving_block_bootstrap(
                anchor_diff,
                BLOCK_ANCHORS,
                N_BOOT,
                seed=(
                    DATASET_SEED[
                        dataset_name
                    ]
                    +
                    H * 100
                    +
                    len(
                        bootstrap_rows
                    )
                ),
            )

            result.update({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Baseline":
                    baseline,

                "Proposed":
                    "Learned",
            })

            bootstrap_rows.append(
                result
            )

bootstrap = pd.DataFrame(
    bootstrap_rows
)

display(bootstrap)

bootstrap.to_csv(
    OUT_DIR /
    "04_similarity_bootstrap.csv",
    index=False,
)


## 11. Dataset-level summary across horizons

In [ ]:

dataset_rows = []

for dataset_name in DATASETS:
    x = paper_table[
        paper_table[
            "Dataset"
        ] ==
        dataset_name
    ]

    b = bootstrap[
        bootstrap[
            "Dataset"
        ] ==
        dataset_name
    ]

    row = {
        "Dataset":
            dataset_name,

        "OursBeatsBestConventional_Tasks":
            int(
                (
                    x[
                        "Learned"
                    ]
                    <
                    x[
                        "BestConventionalSimilarity"
                    ]
                ).sum()
            ),

        "Mean_Ours_vs_BestConventional_%":
            float(
                x[
                    "Ours_vs_BestConventional_%"
                ].mean()
            ),
    }

    for baseline in BASELINES_FOR_BOOTSTRAP:
        bb = b[
            b[
                "Baseline"
            ] ==
            baseline
        ]

        row[
            f"OursBeats_{baseline}_SigHorizons"
        ] = int(
            bb[
                "SignificantImprovement"
            ].sum()
        )

    dataset_rows.append(
        row
    )

dataset_summary = pd.DataFrame(
    dataset_rows
)

display(dataset_summary)

dataset_summary.to_csv(
    OUT_DIR /
    "05_dataset_similarity_summary.csv",
    index=False,
)


## 12. Automatic robustness check

In [ ]:

n_tasks = len(
    paper_table
)

check = pd.DataFrame([
    {
        "Tasks":
            n_tasks,

        "OursBeatsRawCosine":
            int(
                (
                    paper_table[
                        "Learned"
                    ]
                    <
                    paper_table[
                        "RawCosine"
                    ]
                ).sum()
            ),

        "OursBeatsPatternPearson":
            int(
                (
                    paper_table[
                        "Learned"
                    ]
                    <
                    paper_table[
                        "PatternPearson"
                    ]
                ).sum()
            ),

        "OursBeatsNegativeL2":
            int(
                (
                    paper_table[
                        "Learned"
                    ]
                    <
                    paper_table[
                        "NegativeL2"
                    ]
                ).sum()
            ),

        "OursBeatsSpectralCosine":
            int(
                (
                    paper_table[
                        "Learned"
                    ]
                    <
                    paper_table[
                        "SpectralCosine"
                    ]
                ).sum()
            ),

        "OursBeatsSARAFMatched":
            int(
                (
                    paper_table[
                        "Learned"
                    ]
                    <
                    paper_table[
                        "SARAFMatched"
                    ]
                ).sum()
            ),

        "OursBeatsPostHocBestConventional":
            int(
                (
                    paper_table[
                        "Learned"
                    ]
                    <
                    paper_table[
                        "BestConventionalSimilarity"
                    ]
                ).sum()
            ),
    }
])

display(check)

check.to_csv(
    OUT_DIR /
    "06_similarity_diagnostic_check.csv",
    index=False,
)



# Interpretation guide

A natural robustness question is:

> Perhaps cosine/Pattern is simply a weak similarity metric.

The experiment should be interpreted at three levels.

### 1. Which fixed similarity is strongest?

Compare:

- Raw Cosine
- Pattern/Pearson
- Negative L2
- Spectral Cosine

It is completely acceptable if the strongest conventional similarity changes by dataset or horizon. In fact, that would reinforce the paper's argument that there is no single universally reliable surface-similarity notion.

### 2. Does Ours beat each conventional similarity?

This is the primary robustness test.

If Ours remains better across most or all tasks, we can state:

> The gain is not specific to the centered-cosine baseline. Future-compatible relevance learning remains stronger than conventional level-, shape-, and frequency-based retrieval similarities under the same temporally admissible memory.

### 3. Does Ours beat the post-hoc best conventional similarity?

`BestConventionalSimilarity` is intentionally optimistic because it uses the best test-time conventional metric per task.

If Ours still beats this lower envelope broadly, that is particularly strong evidence that the proposed gain is not caused by choosing a weak similarity baseline.

Do **not** tune the proposed model after seeing these results.

This completes the planned similarity-robustness diagnostic.
